# Week 89, Day 6: Sprint review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w089d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
tests = {"passed": 214, "failed": 0, "skipped": 2}
demo = ["PASS 1: waymark plan Mill Ridge", "PASS 2: waymark plan Mill Moon"]
print(tests["failed"] == 0 and all(line.startswith("PASS") for line in demo))

**Predict before you run:** what will this print?



In [ ]:
old = {"Mill to Ridge": 7, "grid corner": 900, "round walk": 97}
new = {"Mill to Ridge": 4, "grid corner": 890, "round walk": 160}
print({k: round(new[k] / old[k], 2) for k in old})

**Predict before you run:** what will this print?



In [ ]:
import math
history = [8, 6, 13]
recent = history[-3:]
print(round(sum(recent) / 3, 1), math.ceil(24 / (sum(recent) / 3)), math.ceil(24 / min(recent)))

**Predict before you run:** what will this print?



In [ ]:
findings = ["complexity 9: pick", "no docstring: astar", "test smell (sleeps): test_pick"]
print(sum(f.startswith(("complexity", "long", "test smell")) for f in findings), 100 - 10 * 2 - 5)

**Predict before you run:** what will this print?



In [ ]:
blocking = []
warnings = ["2 skipped tests", "quality score 75"]
print("GO" if not blocking else "NO GO", len(warnings))

### Exercise: A* on a grid

Write `astar_km(paths, positions, start, end)`: A* returning only the shortest distance, rounded to 2 places (or `None` if unreachable), with `paths` a dict of dicts of km and `positions` a dict of `(x, y)` in km; use `math.dist` for the heuristic.

In [ ]:
import heapq
import math

def astar_km(paths, positions, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_astar():\n    \"a small grid\"\n    positions = {\"A\": (0, 0), \"B\": (1, 0), \"C\": (1, 1), \"D\": (0, 1)}\n    paths = {\"A\": {\"B\": 1.0, \"D\": 1.2}, \"B\": {\"A\": 1.0, \"C\": 1.5}, \"C\": {\"B\": 1.5, \"D\": 1.0}, \"D\": {\"A\": 1.2, \"C\": 1.0}}\n    assert astar_km(paths, positions, \"A\", \"C\") == 2.2 and astar_km(paths, positions, \"B\", \"D\") == 2.2\n    paths[\"E\"] = {}\n    positions[\"E\"] = (5, 5)\n    assert astar_km(paths, positions, \"A\", \"E\") is None")

### Exercise: One more test smell

Write `asserts_in_loops(source)`: the names of test functions (names starting with `test`) in which every `assert` statement is inside a `for` or `while` loop and there's at least one assert (such tests pass silently if the loop runs zero times), in order.

In [ ]:
import ast

def asserts_in_loops(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loops():\n    \"assertions that may never run\"\n    src = '''def test_a(routes):\n    for r in routes:\n        assert r.km > 0\n\ndef test_b(routes):\n    assert routes\n    for r in routes:\n        assert r.km > 0\n\ndef test_c():\n    assert 1 == 1\n\ndef helper():\n    for x in []:\n        assert x\n'''\n    assert asserts_in_loops(src) == [\"test_a\"]")

### Exercise: A go/no-go check

Write `release_check(evidence, quality_limit=80)`. `evidence` is a dict with `tests` (`{"passed", "failed", "skipped"}`), `demo` (report lines starting `PASS` or `FAIL`), `regressions` (a list of changed case names) and `accepted` (case names whose change was accepted on purpose), `blocking_comments` (a number), `perf` (a dict from benchmark name to the ratio new work / old work), `perf_limit` (the largest acceptable ratio), `quality` (a score) and `carried` (story ids). Return a dict with `verdict` (`"GO"` or `"NO GO"`), `blocking` and `warnings` (lists of strings, in the order below), and `text`. Blocking, in order: `"N failing tests"`; `"demo failed: STEP"` for each failing demo line (the text after `FAIL `); `"unexplained change: NAME"` for each regression not accepted; `"N open blocking comments"`; `"slower: NAME (xR)"` for each benchmark above the limit (sorted by name, ratio to 2 places). Warnings, in order: `"N skipped tests"`; `"quality score S below Q"`; `"carried over: S4, S9"`; `"slower but within limit: NAME (xR)"` for ratios above 1 and up to the limit (sorted). Counts appear only when above 0. `text` is `"VERDICT"` then a line per blocking reason prefixed `"- "` and then per warning prefixed `"~ "`.

In [ ]:
def release_check(evidence, quality_limit=80):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = {\"tests\": {\"passed\": 214, \"failed\": 0, \"skipped\": 2}, \"demo\": [\"PASS 1: waymark plan Mill Ridge\", \"PASS 2: waymark walks\"],\n        \"regressions\": [\"S1 shortest to the ridge\"], \"accepted\": [\"S1 shortest to the ridge\"], \"blocking_comments\": 0,\n        \"perf\": {\"route search\": 0.57, \"round walk\": 1.1, \"import\": 1.0}, \"perf_limit\": 1.25, \"quality\": 75, \"carried\": [\"S10\"]}\n\ndef test_go():\n    \"release, with warnings\"\n    r = release_check(GOOD)\n    assert r[\"verdict\"] == \"GO\" and r[\"blocking\"] == []\n    assert r[\"warnings\"] == [\"2 skipped tests\", \"quality score 75 below 80\", \"carried over: S10\", \"slower but within limit: round walk (x1.1)\"]\n    assert r[\"text\"] == \"GO\\n~ 2 skipped tests\\n~ quality score 75 below 80\\n~ carried over: S10\\n~ slower but within limit: round walk (x1.1)\"\n\ndef test_no_go():\n    \"every reason, in order\"\n    bad = dict(GOOD, tests={\"passed\": 210, \"failed\": 3, \"skipped\": 0}, demo=[\"PASS 1: a\", \"FAIL 2: waymark plan Mill Moon: output differs\"],\n               regressions=[\"S1 shortest to the ridge\", \"S9 flattest to the ridge\"], blocking_comments=2,\n               perf={\"round walk\": 1.65, \"route search\": 0.9}, quality=90, carried=[])\n    r = release_check(bad)\n    assert r[\"verdict\"] == \"NO GO\"\n    assert r[\"blocking\"] == [\"3 failing tests\", \"demo failed: 2: waymark plan Mill Moon: output differs\",\n                             \"unexplained change: S9 flattest to the ridge\", \"2 open blocking comments\", \"slower: round walk (x1.65)\"]\n    assert r[\"warnings\"] == [] and r[\"text\"].startswith(\"NO GO\\n- 3 failing tests\\n\")")

> **Waymark's second sprint, in numbers.** Sprint 2 finished 13 points (S5 carried in, S7 and S9), with S4's round walks done but its growth test showing 160 searches where 97 were planned, inside the 1.25 limit only after a fix in review. The forecast for the remaining 24 points is two sprints (three in the worst case), with one build sprint left: S10 (export) stays cut, and S11 (a walk's elevation profile chart) becomes a "could". The release check said GO with four warnings, and the quality score of 75 is the one to fix first: it's cheap, and the hardening week depends on it. What would your release check say today?

### Exercise: Bonus: benchmarks between versions

Write `compare_benchmarks(old, new, limit=1.25)`: `old` and `new` map scenario names to work counts. Return a dict with `ratios` (each scenario in both, sorted by name, to `new / old` rounded to 2 places), `regressions` (names with a ratio above `limit`, sorted by ratio descending then name), `improvements` (ratio at most 0.8, sorted by ratio ascending then name), `added` and `removed` (sorted names in only `new` or only `old`), and `report`: lines `"NAME: OLD -> NEW (xRATIO)"` for every compared scenario in name order, each followed by `" REGRESSION"` or `" improved"` where it applies, then `"added: A, B"` and `"removed: C"` lines if any.

In [ ]:
def compare_benchmarks(old, new, limit=1.25):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_compare():\n    \"version against version\"\n    old = {\"Mill to Ridge\": 7, \"grid corner\": 900, \"round walk\": 97, \"import 1000\": 1001}\n    new = {\"Mill to Ridge\": 4, \"grid corner\": 890, \"round walk\": 160, \"flattest route\": 12}\n    r = compare_benchmarks(old, new)\n    assert r[\"ratios\"] == {\"Mill to Ridge\": 0.57, \"grid corner\": 0.99, \"round walk\": 1.65}\n    assert r[\"regressions\"] == [\"round walk\"] and r[\"improvements\"] == [\"Mill to Ridge\"]\n    assert r[\"added\"] == [\"flattest route\"] and r[\"removed\"] == [\"import 1000\"]\n    assert r[\"report\"] == [\"Mill to Ridge: 7 -> 4 (x0.57) improved\", \"grid corner: 900 -> 890 (x0.99)\",\n                           \"round walk: 97 -> 160 (x1.65) REGRESSION\", \"added: flattest route\", \"removed: import 1000\"]\n    same = compare_benchmarks({\"a\": 10}, {\"a\": 10})\n    assert same[\"report\"] == [\"a: 10 -> 10 (x1.0)\"] and same[\"regressions\"] == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Fit to release?

A release decision is a small design problem: decide **in advance** what evidence a release needs, so the decision on the day is a check, not a debate. Waymark's rules:

- **Blocking** (no release): any failing test; any failing demo step; any **unexplained** change in the regression suite (a change that hasn't been recorded on purpose); any open blocking review comment; a performance regression above the agreed limit.
- **Warnings** (release, but note them): skipped tests; a quality score below 80; stories carried over; slower but within the limit.

The output is a verdict (**GO** or **NO GO**), the reasons, and the warnings, short enough to read in ten seconds and specific enough to act on. It goes in the sprint review, and in CI it can block a release tag. The same idea scales up to what large projects call a release checklist.

## Bonus: benchmarks between versions

Week 89's growth tests check the shape; a **benchmark suite** checks the numbers, version against version. For a fixed set of scenarios (Mill to Ridge, a grid corner to corner, a two-hour round walk), record the **work** each takes (junctions finished, searches made, queries run) for the released version, and compare the new version's counts with them. A ratio above the agreed limit is a regression to explain or fix; well below 1 is an improvement worth celebrating in the changelog. Counting work rather than time makes the comparison repeatable on any machine, which is what lets it run in CI.

## Weekly quiz

## Week 89 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can measure an algorithm's work, implement A* with an admissible heuristic, cache with version invalidation, and find nearest points with a grid. ([Day 1](w089d1.html))
- I can replace an exponential search with shortest-path trees, banned paths and turning points. ([Day 2](w089d2.html))
- I can express preferences as validated values and costs, find the cheapest route, explain it and offer honest alternatives. ([Day 3](w089d3.html))
- I can test growth, combinations, metamorphic relations and recorded regressions. ([Day 4](w089d4.html))
- I can review complexity, interfaces and tests with measurements from the syntax tree. ([Day 5](w089d5.html))
- I can decide whether a version is fit to release, and compare benchmarks between versions. (Today)

## Your capstone today

1. **Your release rules**: write down what blocks a release and what's a warning, and implement `release_check` for your project's evidence.
2. **A benchmark suite**: three to five scenarios with work counts, recorded for this release and compared in CI from now on.
3. **The numbers**: velocity, forecast, and what's cut. One build sprint remains: everything planned for it must be a must or a should you're confident of.
4. **Release** if the check says GO; fix what blocks it if not.

Tomorrow is a **rest day**. Next week is **Capstone 4: the last build sprint**, which finishes Waymark's features: statistics with charts, sharing routes, and the API completed.